In [7]:
from langgraph.graph import StateGraph,START,END
from typing import TypedDict,Literal,Annotated
from langchain_core.messages import HumanMessage,BaseMessage
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.checkpoint.memory import MemorySaver
load_dotenv()

True

In [8]:
from langgraph.graph import add_messages
class ChatState(TypedDict):
    messages:Annotated[list[BaseMessage], add_messages]

In [9]:
llm=ChatOpenAI()
def chat_node(state: ChatState):
    #take user query from state
    messages = state['messages']
    #give it to llm
    response=llm.invoke(messages)
    #response store state
    return {"messages": [response]}
    

In [10]:
checkpointer=MemorySaver()
graph = StateGraph(ChatState)

# add nodes
graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [11]:
thread_id='1'
while True:
    user_message=input("Type Here")
    print("User:", user_message)
    if user_message.lower() in ['exit', 'quit','bye']:
        break
    config={'configurable':{'thread_id': thread_id}}
    result = chatbot.invoke({"messages": [HumanMessage(content=user_message)]}, config=config)
    print('AI:',result['messages'][-1].content)

User: hi my name is piwi
AI: Hi Piwi, nice to meet you! How can I assist you today?
User: what is my name
AI: Your name is Piwi.
User: exit
